In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/road_fine.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "dismissal": "string",
        "vehicleClass": "string",
        "notificationType": "string",
        "lastSent": "string",
        "amount": "float32",
        "totalPaymentAmount": "float32",
        "article": "float32",
        "points": "float32",
        "expense": "float32",
        "paymentAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,amount,article,concept:name,dismissal,expense,lastSent,lifecycle:transition,notificationType,org:resource,paymentAmount,points,time_delta,totalPaymentAmount,vehicleClass
0,A1,2006-07-24,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
1,A1,2006-12-05,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11577600.0,0.0,NA
2,A100,2006-08-02,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
3,A100,2006-12-12,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11404800.0,0.0,NA
4,A100,2007-01-15,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,2937600.0,0.0,NA
5,A100,2007-03-16,71.5,0.0,Add penalty,NA,0.0,NA,complete,NA,NA,0.0,0.0,5184000.0,0.0,NA
6,A100,2009-03-30,0.0,0.0,Send for Credit Collection,NA,0.0,NA,complete,NA,NA,0.0,0.0,64368000.0,0.0,NA
7,A10000,2007-03-09,36.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
8,A10000,2007-07-17,0.0,0.0,Send Fine,NA,13.0,NA,complete,NA,NA,0.0,0.0,11232000.0,0.0,NA
9,A10000,2007-08-02,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,1382400.0,0.0,NA


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['amount', 'article', 'concept:name', 'dismissal', 'expense', 'lastSent', 'lifecycle:transition', 'notificationType', 'org:resource', 'paymentAmount', 'points', 'time_delta', 'totalPaymentAmount', 'vehicleClass']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 8726400.00]                       2160000.0000 quantile_derived    
amount                         continuous     event    yes    [24.00, 80.00]                           6.7000     quantile_derived    
totalPaymentAmount             continuous     event

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Payment',
  'Send Fine'},
 {'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Send Fine'}]

In [13]:
engine.branching_sets

[{'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Payment',
  'Send Fine',
  'Send for Credit Collection'},
 {'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Send Fine'},
 {'Appeal to Judge', 'Insert Fine Notification', 'Send Fine'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/road_fine-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/310 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,A19080,5,1,4,0.412940,0.245879,0.58000,0.504167,0.000000,...,0.340693,0.000000,0.132359,0.200,0.064719,0.208333,0.0,0.0,0.0,0.0
1,0,S106832,6,1,5,0.448291,0.291583,0.60500,0.539583,0.093333,...,0.589235,0.000000,0.255902,0.400,0.111803,0.333333,0.0,0.0,0.0,0.0
2,0,S160509,7,1,4,0.536960,0.473920,0.60000,0.639583,0.352941,...,0.934640,0.352941,0.248366,0.300,0.196731,0.333333,0.0,0.0,0.0,0.0
3,1,S71772,3,1,2,0.394200,0.273401,0.51500,0.456250,0.000000,...,0.135017,0.000000,0.051684,0.000,0.103367,0.083333,0.0,0.0,0.0,0.0
4,1,V6556,3,1,2,0.419476,0.293951,0.54500,0.458333,0.000000,...,0.169048,0.000000,0.085714,0.100,0.071429,0.083333,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,29,S54626,9,1,4,0.359687,0.268123,0.45125,0.464062,0.000000,...,0.196774,0.000000,0.061357,0.050,0.072715,0.135417,0.0,0.0,0.0,0.0
222,30,N53905,9,1,4,0.342039,0.216579,0.46750,0.422917,0.480952,...,0.699918,0.523810,0.061525,0.050,0.073050,0.114583,0.0,0.0,0.0,0.0
223,30,A373,9,1,4,0.360223,0.242946,0.47750,0.437500,0.250000,...,0.121721,0.000000,0.038388,0.025,0.051775,0.083333,0.0,0.0,0.0,0.0
224,30,N56915,9,1,4,0.453032,0.329814,0.57625,0.551042,0.092857,...,0.189747,0.000000,0.064747,0.050,0.079493,0.125000,0.0,0.0,0.0,0.0


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/350 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,A19080,5,2,4,0.505915,0.441829,0.570000,0.608333,0.187500,...,0.807325,0.187500,0.244825,0.200000,0.289650,0.375000,0.0,0.488010,0.0,0.500000
1,0,N63824,6,2,5,0.529993,0.354986,0.705000,0.585417,0.277778,...,0.933803,0.277778,0.281025,0.300000,0.262050,0.375000,0.0,0.488850,0.0,0.500000
2,0,N83410,7,2,6,0.399877,0.364753,0.435000,0.475000,0.350000,...,0.706983,0.350000,0.148650,0.100000,0.197299,0.208333,0.0,0.487989,0.0,0.500000
3,0,A31097,8,2,4,0.467465,0.329930,0.605000,0.539583,0.668182,...,0.635060,0.272727,0.153999,0.100000,0.207998,0.208333,0.0,0.489314,0.0,0.500000
4,1,N100658,4,2,2,0.388472,0.216944,0.560000,0.410417,0.000000,...,0.091667,0.000000,0.050000,0.100000,0.000000,0.041667,0.0,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,33,N47801,10,3,5,0.356631,0.261595,0.451667,0.430556,0.486207,...,0.656692,0.482759,0.062822,0.033333,0.092311,0.111111,0.0,0.634577,0.0,0.666667
268,33,C16950,12,3,7,0.284764,0.166194,0.403333,0.323611,0.606061,...,0.637121,0.545455,0.050000,0.100000,0.000000,0.041667,0.0,0.629122,0.0,0.666667
269,34,S48577,9,3,3,0.347270,0.224540,0.470000,0.400000,0.481481,...,0.514481,0.444444,0.028370,0.033333,0.023407,0.041667,0.0,0.909268,0.0,1.000000
270,34,S76033,9,3,3,0.322045,0.225757,0.418333,0.393056,0.481481,...,0.649379,0.444444,0.079934,0.066667,0.093202,0.125000,0.0,0.647563,0.0,0.666667


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()